# German Neural Reader – Google Colab\nLädt eine TXT-Datei hoch, berechnet die deutsche Stimme Eva K extern in Colab und erzeugt eine MP3.

In [ ]:
!apt-get -qq update\n!apt-get -qq install -y ffmpeg\n!pip -q install piper-tts==1.3.0

In [ ]:
import urllib.request, subprocess, pathlib\nfrom google.colab import files\n\nMODEL='de_DE-eva_k-x_low.onnx'\nCONFIG='de_DE-eva_k-x_low.onnx.json'\nBASE='https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/de/de_DE/eva_k/x_low/'\nurllib.request.urlretrieve(BASE+MODEL+'?download=true', MODEL)\nurllib.request.urlretrieve(BASE+CONFIG+'?download=true', CONFIG)\nprint('Eva K geladen.')

In [ ]:
uploaded = files.upload()\nname = next(iter(uploaded))\ntext = uploaded[name].decode('utf-8')\nprint(f'{len(text):,} Zeichen geladen.')

In [ ]:
SPEED = 0.94\nBITRATE = 40\nSENTENCE_PAUSE = 0.24\n\ncmd = [\n    'piper','--model',MODEL,'--config',CONFIG,\n    '--output_file','GermanReader.wav',\n    '--length_scale',str(1.0/SPEED),\n    '--sentence_silence',str(SENTENCE_PAUSE)\n]\np = subprocess.run(cmd, input=text, text=True, capture_output=True)\nprint(p.stderr[-1000:])\nif p.returncode != 0:\n    raise RuntimeError('Piper failed')\n\nsubprocess.run([\n    'ffmpeg','-y','-hide_banner','-loglevel','error',\n    '-i','GermanReader.wav','-codec:a','libmp3lame',\n    '-b:a',f'{BITRATE}k','GermanReader.mp3'\n], check=True)\nprint('MP3 fertig.')

In [ ]:
files.download('GermanReader.mp3')